# Notebook 06 — Full Semi-DETR-style: SHM + CQC + CPM, 150 epochs — Standardized

Notebook này viết tiếp từ **NB05** bằng cách thêm **Cost-based Pseudo Label Mining (CPM)**.

Vai trò thí nghiệm:
- **Không load checkpoint NB02**: giữ setup độc lập giống NB03A/NB04/NB05.
- Dữ liệu: 10% labeled + 90% unlabeled.
- Module: **SHM + CQC + CPM**.
- Đây là bản **Full Semi-DETR-style / HF-compatible implementation**, vì SHM/CQC/CPM được triển khai dạng approximation/proxy để chạy với `DeformableDetrForObjectDetection`.

Tên báo cáo nên dùng: **Full Semi-DETR-style: SHM + CQC + CPM**.

> Aligned CPM single-step variant: starts from the standardized NB06 but sets `unlabeled_steps_per_labeled=1` because CPM was unstable in the 2-unlabeled-step run. All other standardized pieces are kept.


In [ ]:
# Chạy cell này trên Kaggle/Colab nếu môi trường thiếu package.
import sys, subprocess, importlib.util

REQUIRED_PACKAGES = [
    ("transformers", "transformers==4.40.2"),
    ("timm", "timm"),
    ("pycocotools", "pycocotools"),
    ("albumentations", "albumentations>=1.3.0"),
    ("sklearn", "scikit-learn"),
]

def ensure_package(import_name, pip_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in REQUIRED_PACKAGES:
    ensure_package(import_name, pip_name)

print("Dependencies are ready")

In [ ]:
import os, io, json, time, math, copy, random, contextlib, warnings, tempfile
from pathlib import Path
from collections import Counter, defaultdict

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"  # --- IGNORE ---
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from transformers import AutoImageProcessor, DeformableDetrConfig, DeformableDetrForObjectDetection
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

try:
    from torchvision.ops import nms
except Exception:
    nms = None

try:
    from sklearn.mixture import GaussianMixture
except Exception:
    GaussianMixture = None


print(f"Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def resolve_data_root(candidates):
    for cand in candidates:
        root = Path(cand)
        if (root / "train/images").exists() and (root / "train/labels").exists():
            return root
    raise FileNotFoundError(
        "Không tìm thấy data_root hợp lệ. Hãy sửa CFG['data_root_candidates'] cho đúng Kaggle dataset path.\n"
        + "Candidates:\n" + "\n".join(map(str, candidates))
    )


def scan_split(data_root: Path, split: str):
    img_dir = data_root / split / "images"
    lbl_dir = data_root / split / "labels"
    if not img_dir.exists():
        raise FileNotFoundError(f"Missing image folder: {img_dir}")
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    img_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in exts])
    if len(img_paths) == 0:
        raise RuntimeError(f"No images found in: {img_dir}")
    return [(p, lbl_dir / f"{p.stem}.txt") for p in img_paths]


def split_labeled_unlabeled(pairs, ratio: float, seed: int):
    rng = random.Random(seed)
    pairs = list(pairs)
    rng.shuffle(pairs)
    n_labeled = max(1, int(round(len(pairs) * ratio)))
    return pairs[:n_labeled], pairs[n_labeled:]


def read_raw_yolo_classes(label_path: Path):
    classes = []
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    classes.append(int(float(parts[0])))
    return classes


def infer_label_offset(pairs, num_classes: int):
    raw = []
    for _, lbl in pairs:
        raw.extend(read_raw_yolo_classes(lbl))
    if not raw:
        return 0, {"min": None, "max": None, "counter": {}}
    mn, mx = min(raw), max(raw)
    # Hỗ trợ cả YOLO class 0..C-1 và 1..C.
    offset = 1 if (mn == 1 and mx == num_classes) else 0
    return offset, {"min": int(mn), "max": int(mx), "counter": dict(Counter(raw))}


def load_yolo_label(label_path: Path, img_w: int, img_h: int, label_offset: int, num_classes: int):
    boxes, labels = [], []
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(float(parts[0])) - label_offset
                if cls < 0 or cls >= num_classes:
                    continue
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw / 2.0) * img_w
                y1 = (cy - bh / 2.0) * img_h
                x2 = (cx + bw / 2.0) * img_w
                y2 = (cy + bh / 2.0) * img_h
                x1, y1 = max(0.0, x1), max(0.0, y1)
                x2, y2 = min(float(img_w), x2), min(float(img_h), y2)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls)
    return np.asarray(boxes, dtype=np.float32), np.asarray(labels, dtype=np.int64)


def make_pad_if_needed(img_size: int):
    # Albumentations 1.x dùng value, 2.x ưu tiên fill. Try/except để notebook chạy ổn trên nhiều Kaggle image.
    try:
        return A.PadIfNeeded(
            min_height=img_size,
            min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
        )
    except TypeError:
        return A.PadIfNeeded(
            min_height=img_size,
            min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
        )


def make_coarse_dropout():
    try:
        return A.CoarseDropout(max_holes=8, max_height=64, max_width=64, p=0.25)
    except TypeError:
        return A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(8, 64),
            hole_width_range=(8, 64),
            p=0.25,
        )


def get_transform(img_size: int, train: bool):
    # Không dùng rotation thủ công vì dễ làm sai bbox. Tất cả transform hình học đi qua Albumentations bbox-aware.
    transforms = [
        A.LongestMaxSize(max_size=img_size),
        make_pad_if_needed(img_size),
    ]
    if train:
        transforms.extend([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.RandomBrightnessContrast(p=0.35),
            A.HueSaturationValue(p=0.20),
        ])
    return A.Compose(
        transforms,
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["class_labels"],
            min_visibility=0.10,
            check_each_transform=True,
        ),
    )


def get_strong_photometric_aug():
    # Chuẩn hóa across NB03A/NB04/NB05/NB06: PCB defects thường nhỏ; aggressive blur/cutout
    # có thể làm mất defect nhưng pseudo-box vẫn còn. Giữ strong view chủ yếu là photometric nhẹ.
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.55),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.25),
        A.GaussNoise(p=0.10),
        A.MotionBlur(blur_limit=3, p=0.05),
    ])

def xyxy_to_cxcywh_norm(boxes, width: int, height: int):
    if len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    boxes = np.asarray(boxes, dtype=np.float32)
    cx = (boxes[:, 0] + boxes[:, 2]) / 2.0 / width
    cy = (boxes[:, 1] + boxes[:, 3]) / 2.0 / height
    bw = (boxes[:, 2] - boxes[:, 0]) / width
    bh = (boxes[:, 3] - boxes[:, 1]) / height
    return np.clip(np.stack([cx, cy, bw, bh], axis=1), 0.0, 1.0).astype(np.float32)


def cxcywh_norm_to_xyxy_abs(boxes, img_size: int):
    boxes = np.asarray(boxes, dtype=np.float32)
    if len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    cx, cy, bw, bh = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x1 = (cx - bw / 2.0) * img_size
    y1 = (cy - bh / 2.0) * img_size
    x2 = (cx + bw / 2.0) * img_size
    y2 = (cy + bh / 2.0) * img_size
    out = np.stack([x1, y1, x2, y2], axis=1)
    return np.clip(out, 0.0, float(img_size)).astype(np.float32)


class YoloDetectionDataset(Dataset):
    def __init__(self, pairs, img_size, transform, label_offset, num_classes, mode="train"):
        self.pairs = list(pairs)
        self.img_size = int(img_size)
        self.transform = transform
        self.label_offset = int(label_offset)
        self.num_classes = int(num_classes)
        self.mode = mode

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise FileNotFoundError(img_path)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        h0, w0 = image.shape[:2]
        boxes, labels = load_yolo_label(lbl_path, w0, h0, self.label_offset, self.num_classes)
        aug = self.transform(image=image, bboxes=boxes.tolist(), class_labels=labels.tolist())
        image = aug["image"]
        boxes = np.asarray(aug["bboxes"], dtype=np.float32)
        labels = np.asarray(aug["class_labels"], dtype=np.int64)
        h, w = image.shape[:2]
        boxes_norm = xyxy_to_cxcywh_norm(boxes, w, h)
        return {
            "image": image,
            "boxes": boxes_norm,
            "labels": labels,
            "image_id": int(idx),
            "img_path": str(img_path),
        }


class YoloUnlabeledDataset(Dataset):
    def __init__(self, pairs, img_size, weak_transform, strong_photo_aug):
        self.pairs = list(pairs)
        self.img_size = int(img_size)
        self.weak_transform = weak_transform
        self.strong_photo_aug = strong_photo_aug

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, _ = self.pairs[idx]
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise FileNotFoundError(img_path)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        weak_aug = self.weak_transform(image=image, bboxes=[], class_labels=[])
        weak = weak_aug["image"]
        strong = self.strong_photo_aug(image=weak)["image"]
        return {
            "weak_image": weak,
            "strong_image": strong,
            "image_id": int(idx),
            "img_path": str(img_path),
        }


def sample_to_coco_annotations(item, img_size: int, ann_id_start=1):
    anns = []
    for j, (box, cls) in enumerate(zip(item["boxes"], item["labels"])):
        cx, cy, bw, bh = map(float, box)
        x1 = max(0.0, (cx - bw / 2.0) * img_size)
        y1 = max(0.0, (cy - bh / 2.0) * img_size)
        x2 = min(float((cx + bw / 2.0) * img_size), float(img_size))
        y2 = min(float((cy + bh / 2.0) * img_size), float(img_size))
        w, h = x2 - x1, y2 - y1
        if w <= 1 or h <= 1:
            continue
        anns.append({
            "id": int(ann_id_start + j),
            "image_id": int(item["image_id"]),
            "category_id": int(cls),
            "bbox": [float(x1), float(y1), float(w), float(h)],
            "area": float(w * h),
            "iscrowd": 0,
        })
    return anns


def seed_worker(worker_id):
    worker_seed = CFG["seed"] + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def save_split_report(path, train_pairs, labeled_pairs, unlabeled_pairs, valid_pairs, test_pairs):
    report = {
        "seed": CFG["seed"],
        "labeled_ratio": CFG["labeled_ratio"],
        "train_total": len(train_pairs),
        "labeled": len(labeled_pairs),
        "unlabeled": len(unlabeled_pairs),
        "valid": len(valid_pairs),
        "test": len(test_pairs),
        "labeled_files": [p[0].name for p in labeled_pairs],
        "unlabeled_files": [p[0].name for p in unlabeled_pairs],
    }
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    return report

In [ ]:
CFG = dict(
    seed=42,

    # Hai notebook dùng cùng candidate list và cùng thứ tự. Sửa list này ở CẢ HAI notebook nếu dataset path khác.
    data_root_candidates=[
        "/kaggle/input/datasets/hgthuhwn/my-pcb/YOLO_format",
        "/kaggle/input/datasets/nhnkhnh/my-pcb/YOLO_format",
        "/kaggle/input/datasets/minhquan228/my-pcb/YOLO_format",
        "/kaggle/input/datasets/minhhieu/datapcb-v2/YOLO_format",
        "/kaggle/input/datasets/chungkein/datapcb-v1/YOLO_format",
        "/kaggle/input/my-pcb/YOLO_format",
        "/kaggle/input/datapcb-v2/YOLO_format",
    ],
    out_dir="/kaggle/working/artifacts/nb06_full_semi_detr_shm_cqc_cpm_aligned_single_step_150e",
    model_name="SenseTime/deformable-detr",
    num_classes=5,
    class_names=["0", "1", "2", "3", "4"],
    num_queries=300,
    num_feature_levels=4,
    img_size=640,

    labeled_ratio=0.10,
    epochs=150,
    burn_in_epochs=80,
    batch_size=1,
    unlabeled_batch_size=1,
    unlabeled_steps_per_labeled=1,   # CPM-stable setting: one unlabeled batch per labeled batch.
    grad_accum_steps=16,      # effective labeled batch = 16, giống baseline.
    num_workers=2,
    pin_memory=True,
    mixed_precision=True,

    lr=2e-4,
    lr_backbone=2e-5,
    weight_decay=1e-4,
    lr_milestones=[105, 135],
    lr_gamma=0.1,
    clip_grad_norm=0.1,
    ema_momentum=0.995,
    pseudo_threshold_start=0.50,
    pseudo_threshold_final=0.25,
    pseudo_threshold_warmup_epochs=60,
    max_pseudo_per_image=60,
    min_box_size_px=2.0,
    nms_iou_threshold=0.65,
    unsup_weight_start=0.00,
    unsup_weight_final=0.35,
    unsup_weight_warmup_epochs=70,
    # Stage 0: burn-in supervised O2O để teacher đủ ổn định.
    # Stage 1: O2M approximation bằng cách repeat GT/pseudo boxes.
    # Stage 2: quay lại O2O chuẩn của Deformable DETR.
    shm_stage1_epochs=120,
    o2m_repeat_factor=2,
    max_o2m_targets_per_image=240,
    shm_apply_to_supervised=False,
    shm_apply_to_unsupervised=True,
    # Đây là CQC proxy để chạy được với HuggingFace Deformable DETR.
    # Loss được ramp-up sau burn-in để tránh pseudo-label nhiễu gây bất ổn sớm.
    cqc_weight_start=0.00,
    cqc_weight_final=0.15,
    cqc_weight_warmup_epochs=70,
    cqc_max_boxes_per_image=30,
    cqc_cls_weight=1.0,
    cqc_box_weight=2.0,
    cqc_match_cls_cost=1.0,
    cqc_match_l1_cost=2.0,
    cqc_match_iou_cost=2.0,

    # Cost-based Pseudo Label Mining (CPM) for CQC anchors.
    # Paper-level CPM uses matching cost distribution and GMM to mine reliable pseudo boxes.
    # This HF-compatible version computes class/L1/IoU cost between pseudo boxes and student queries.
    cpm_enabled=True,
    cpm_min_gmm_samples=4,
    cpm_strategy="gmm_low_cluster",       # options: gmm_low_cluster, reliable_center, midpoint, quantile
    cpm_fallback_quantile=0.60,
    cpm_keep_at_least_one_per_image=True,
    cpm_cls_cost=1.0,
    cpm_l1_cost=2.0,
    cpm_iou_cost=2.0,

    experiment_name="nb06_full_semi_detr_shm_cqc_cpm_aligned_single_step_150e",
    method_name="full_semi_detr_shm_cqc_cpm_aligned_single_step_independent_10pct_150e",
    enable_adaptive_pseudo=True,
    fallback_min_score=0.20,
    adaptive_min_pseudo_per_image=1,
    adaptive_topk_per_image=6,
    primary_eval_model="student",
    eval_every_n_epochs=5,
    eval_conf_threshold=0.05,
    vis_conf_threshold=0.25,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CFG["device"] = str(DEVICE)
OUT_DIR = Path(CFG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(CFG["seed"])

DATA_ROOT = resolve_data_root(CFG["data_root_candidates"])
CFG["data_root"] = str(DATA_ROOT)

with open(OUT_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print(json.dumps({k: CFG[k] for k in [
    "data_root", "out_dir", "labeled_ratio", "epochs", "burn_in_epochs", "shm_stage1_epochs",
    "o2m_repeat_factor", "cqc_weight_final", "cqc_max_boxes_per_image", "cpm_enabled", "cpm_strategy",
    "batch_size", "unlabeled_batch_size", "grad_accum_steps",
    "lr_milestones", "pseudo_threshold_start", "pseudo_threshold_final", "unsup_weight_final", "device"
]}, indent=2))


In [ ]:
processor = AutoImageProcessor.from_pretrained(
    CFG["model_name"],
    size={"height": CFG["img_size"], "width": CFG["img_size"]},
    do_resize=False,
    use_fast=False,
)


def collate_labeled(batch):
    images, targets = [], []
    ann_id = 1
    for item in batch:
        images.append(Image.fromarray(item["image"]))
        anns = sample_to_coco_annotations(item, CFG["img_size"], ann_id_start=ann_id)
        ann_id += max(1, len(anns))
        targets.append({"image_id": int(item["image_id"]), "annotations": anns})
    enc = processor(images=images, annotations=targets, return_tensors="pt")
    enc["image_ids"] = torch.tensor([int(x["image_id"]) for x in batch], dtype=torch.long)
    return enc


def move_labels_to_device(labels, device):
    return [{k: v.to(device) for k, v in t.items()} for t in labels]


def build_detector(cfg):
    config = DeformableDetrConfig.from_pretrained(cfg["model_name"])
    config.num_labels = cfg["num_classes"]
    config.num_queries = cfg["num_queries"]
    config.num_feature_levels = cfg["num_feature_levels"]
    return DeformableDetrForObjectDetection.from_pretrained(
        cfg["model_name"],
        config=config,
        ignore_mismatched_sizes=True,
    )


def build_optimizer(detector, cfg):
    backbone_params, other_params = [], []
    for name, p in detector.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(p)
        else:
            other_params.append(p)
    return torch.optim.AdamW(
        [
            {"params": other_params, "lr": cfg["lr"]},
            {"params": backbone_params, "lr": cfg["lr_backbone"]},
        ],
        weight_decay=cfg["weight_decay"],
    )


def build_scheduler(optimizer, cfg):
    return torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=list(cfg["lr_milestones"]),
        gamma=cfg["lr_gamma"],
    )


def build_coco_gt_from_dataset(dataset, name="dataset"):
    images, annotations = [], []
    ann_id = 1
    for idx in range(len(dataset)):
        item = dataset[idx]
        images.append({
            "id": int(item["image_id"]),
            "width": CFG["img_size"],
            "height": CFG["img_size"],
            "file_name": Path(item["img_path"]).name,
        })
        anns = sample_to_coco_annotations(item, CFG["img_size"], ann_id_start=ann_id)
        annotations.extend(anns)
        ann_id += max(1, len(anns))
    cats = [{"id": i, "name": str(CFG["class_names"][i])} for i in range(CFG["num_classes"])]
    coco_dict = {
        "info": {"description": name},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": cats,
    }
    gt_path = OUT_DIR / f"gt_{name}.json"
    with open(gt_path, "w") as f:
        json.dump(coco_dict, f)
    with contextlib.redirect_stdout(io.StringIO()):
        return COCO(str(gt_path))


@torch.no_grad()
def evaluate_coco(detector, loader, dataset, device, coco_gt=None, desc="val", conf_thr=None):
    detector.eval()
    conf_thr = CFG["eval_conf_threshold"] if conf_thr is None else float(conf_thr)
    if coco_gt is None:
        with contextlib.redirect_stdout(io.StringIO()):
            coco_gt = build_coco_gt_from_dataset(dataset, name=desc)

    results = []
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch.get("pixel_mask")
        pixel_mask = pixel_mask.to(device) if pixel_mask is not None else None

        with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and device.type == "cuda"):
            outputs = detector(pixel_values=pixel_values, pixel_mask=pixel_mask)

        target_sizes = torch.tensor(
            [[CFG["img_size"], CFG["img_size"]]] * pixel_values.shape[0],
            device=device,
        )
        preds = processor.post_process_object_detection(
            outputs,
            threshold=conf_thr,
            target_sizes=target_sizes,
        )

        for img_id, pred in zip(batch["image_ids"].tolist(), preds):
            boxes = pred["boxes"].detach().cpu().numpy()
            scores = pred["scores"].detach().cpu().numpy()
            labels = pred["labels"].detach().cpu().numpy()
            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = map(float, box)
                x1, y1 = max(0.0, min(x1, CFG["img_size"])), max(0.0, min(y1, CFG["img_size"]))
                x2, y2 = max(0.0, min(x2, CFG["img_size"])), max(0.0, min(y2, CFG["img_size"]))
                w, h = x2 - x1, y2 - y1
                if w <= 1 or h <= 1:
                    continue
                if int(label) < 0 or int(label) >= CFG["num_classes"]:
                    continue
                results.append({
                    "image_id": int(img_id),
                    "category_id": int(label),
                    "bbox": [float(x1), float(y1), float(w), float(h)],
                    "score": float(score),
                })

    empty = {"AP": 0.0, "AP50": 0.0, "AP75": 0.0, "APs": 0.0, "per_class_AP50": {}}
    if not results:
        return empty

    pred_path = OUT_DIR / f"pred_{desc}.json"
    with open(pred_path, "w") as f:
        json.dump(results, f)

    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(str(pred_path))
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
    stats = coco_eval.stats

    per_class_ap50 = {}
    for cat_id in coco_gt.getCatIds():
        cat_name = coco_gt.loadCats(cat_id)[0]["name"]
        with contextlib.redirect_stdout(io.StringIO()):
            coco_cls = COCOeval(coco_gt, coco_dt, "bbox")
            coco_cls.params.catIds = [cat_id]
            coco_cls.params.iouThrs = [0.5]
            coco_cls.evaluate()
            coco_cls.accumulate()
            coco_cls.summarize()
        per_class_ap50[cat_name] = float(coco_cls.stats[0])

    pc_str = "  ".join(f"{k}:{v:.3f}" for k, v in per_class_ap50.items())
    print(f"[{desc}] AP={stats[0]:.4f} AP50={stats[1]:.4f} AP75={stats[2]:.4f} APs={stats[3]:.4f} | {pc_str}")

    return {
        "AP": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "APs": float(stats[3]),
        "per_class_AP50": per_class_ap50,
    }


def safe_torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

In [ ]:
train_pairs = scan_split(DATA_ROOT, "train")
valid_pairs = scan_split(DATA_ROOT, "valid")
test_pairs  = scan_split(DATA_ROOT, "test")

LABEL_OFFSET, label_stats = infer_label_offset(train_pairs, CFG["num_classes"])
CFG["label_offset"] = LABEL_OFFSET

labeled_pairs, unlabeled_pairs = split_labeled_unlabeled(train_pairs, CFG["labeled_ratio"], CFG["seed"])
split_report = save_split_report(OUT_DIR / "split_labeled_10pct_seed42.json", train_pairs, labeled_pairs, unlabeled_pairs, valid_pairs, test_pairs)

print("=" * 72)
print("DATA SPLIT SUMMARY - DETR-SSOD + SHM + CQC")
print("=" * 72)
print(f"Train total     : {len(train_pairs)}")
print(f"Labeled images  : {len(labeled_pairs)} ({len(labeled_pairs) / len(train_pairs) * 100:.2f}%)")
print(f"Unlabeled images: {len(unlabeled_pairs)} - used only for pseudo-label training")
print(f"Valid/Test      : {len(valid_pairs)} / {len(test_pairs)}")
print("Raw label stats :", label_stats)
print("Label offset    :", LABEL_OFFSET)
print("First 5 labeled :", [p[0].name for p in labeled_pairs[:5]])
print("=" * 72)

train_tf = get_transform(CFG["img_size"], train=True)
eval_tf  = get_transform(CFG["img_size"], train=False)
weak_unlabeled_tf = get_transform(CFG["img_size"], train=True)
strong_photo_tf = get_strong_photometric_aug()

labeled_train_ds = YoloDetectionDataset(labeled_pairs, CFG["img_size"], train_tf, LABEL_OFFSET, CFG["num_classes"], mode="train")
unlabeled_train_ds = YoloUnlabeledDataset(unlabeled_pairs, CFG["img_size"], weak_unlabeled_tf, strong_photo_tf)
valid_ds = YoloDetectionDataset(valid_pairs, CFG["img_size"], eval_tf, LABEL_OFFSET, CFG["num_classes"], mode="valid")
test_ds  = YoloDetectionDataset(test_pairs,  CFG["img_size"], eval_tf, LABEL_OFFSET, CFG["num_classes"], mode="test")

labeled_loader = DataLoader(
    labeled_train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_labeled, worker_init_fn=seed_worker, drop_last=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=max(1, CFG["batch_size"] * 2), shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_labeled, worker_init_fn=seed_worker,
)
test_loader = DataLoader(
    test_ds, batch_size=max(1, CFG["batch_size"] * 2), shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_labeled, worker_init_fn=seed_worker,
)

def collate_unlabeled(batch):
    weak_imgs = [Image.fromarray(x["weak_image"]) for x in batch]
    strong_imgs = [Image.fromarray(x["strong_image"]) for x in batch]
    weak = processor(images=weak_imgs, return_tensors="pt")
    strong = processor(images=strong_imgs, return_tensors="pt")
    out = {
        "weak_pixel_values": weak["pixel_values"],
        "strong_pixel_values": strong["pixel_values"],
        "image_ids": torch.tensor([int(x["image_id"]) for x in batch], dtype=torch.long),
    }
    if "pixel_mask" in weak:
        out["weak_pixel_mask"] = weak["pixel_mask"]
    if "pixel_mask" in strong:
        out["strong_pixel_mask"] = strong["pixel_mask"]
    return out

unlabeled_loader = DataLoader(
    unlabeled_train_ds, batch_size=CFG["unlabeled_batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_unlabeled, worker_init_fn=seed_worker, drop_last=True,
)

sample = labeled_train_ds[0]
print("Sample:", sample["image"].shape, sample["boxes"].shape, sample["labels"][:10])
print("Unlabeled images:", len(unlabeled_train_ds))

In [ ]:
def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch


def get_pseudo_threshold(epoch, cfg):
    if epoch <= cfg["burn_in_epochs"]:
        return float(cfg["pseudo_threshold_start"])
    start = float(cfg["pseudo_threshold_start"])
    final = float(cfg["pseudo_threshold_final"])
    warm = max(1, int(cfg["pseudo_threshold_warmup_epochs"]))
    progress = min(1.0, max(0.0, (epoch - cfg["burn_in_epochs"]) / warm))
    return start + progress * (final - start)


def get_unsup_weight(epoch, cfg):
    if epoch <= cfg["burn_in_epochs"]:
        return 0.0
    start = float(cfg["unsup_weight_start"])
    final = float(cfg["unsup_weight_final"])
    warm = max(1, int(cfg["unsup_weight_warmup_epochs"]))
    progress = min(1.0, max(0.0, (epoch - cfg["burn_in_epochs"]) / warm))
    return start + progress * (final - start)


def get_cqc_weight(epoch, cfg):
    """Ramp-up CQC after burn-in, similar to unsupervised loss ramp-up."""
    if epoch <= cfg["burn_in_epochs"]:
        return 0.0
    start = float(cfg.get("cqc_weight_start", 0.0))
    final = float(cfg.get("cqc_weight_final", 0.0))
    warm = max(1, int(cfg.get("cqc_weight_warmup_epochs", 20)))
    progress = min(1.0, max(0.0, (epoch - cfg["burn_in_epochs"]) / warm))
    return start + progress * (final - start)


def get_shm_stage(epoch, cfg):
    """Return training stage for the SHM + CQC + CPM experiment."""
    if epoch <= int(cfg["burn_in_epochs"]):
        return "burn_in_o2o"
    if epoch <= int(cfg["shm_stage1_epochs"]):
        return "o2m+cqc+cpm"
    return "o2o+cqc+cpm"


def repeat_hf_labels_for_o2m(labels, repeat_factor=3, max_targets=None):
    """Approximate one-to-many assignment for HF Deformable DETR.

    HF DeformableDetrForObjectDetection internally uses Hungarian one-to-one matching.
    The original Semi-DETR paper changes the matcher/loss. That is not directly exposed here.

    Practical approximation:
    - duplicate each GT/pseudo object M times in the target list;
    - the internal one-to-one matcher can then assign M different queries to the same object;
    - this gives multiple positive queries per object during Stage 1.
    """
    repeat_factor = int(repeat_factor)
    if repeat_factor <= 1:
        return labels

    out = []
    for target in labels:
        if "boxes" not in target:
            out.append(target)
            continue

        boxes = target["boxes"]
        n = int(boxes.shape[0]) if torch.is_tensor(boxes) and boxes.ndim >= 1 else 0
        if n == 0:
            out.append(target)
            continue

        idx = torch.arange(n, device=boxes.device).repeat_interleave(repeat_factor)
        if max_targets is not None and int(max_targets) > 0 and idx.numel() > int(max_targets):
            idx = idx[:int(max_targets)]

        image_level_keys = {"image_id", "size", "orig_size"}
        new_target = {}
        for key, value in target.items():
            if key in image_level_keys:
                new_target[key] = value
            elif torch.is_tensor(value) and value.ndim >= 1 and value.shape[0] == n:
                new_target[key] = value[idx]
            else:
                new_target[key] = value
        out.append(new_target)
    return out


def maybe_apply_shm_o2m(labels, epoch, cfg, apply=True):
    if not apply:
        return labels
    if not get_shm_stage(epoch, cfg).startswith("o2m"):
        return labels
    return repeat_hf_labels_for_o2m(
        labels,
        repeat_factor=cfg.get("o2m_repeat_factor", 3),
        max_targets=cfg.get("max_o2m_targets_per_image", None),
    )


def count_targets(labels):
    total = 0
    for target in labels:
        if "boxes" in target and torch.is_tensor(target["boxes"]):
            total += int(target["boxes"].shape[0])
    return total


def xyxy_abs_to_cxcywh_norm_torch(boxes_xyxy, img_size):
    if boxes_xyxy.numel() == 0:
        return boxes_xyxy.new_zeros((0, 4))
    boxes = boxes_xyxy.clamp(0, img_size) / float(img_size)
    x1, y1, x2, y2 = boxes.unbind(-1)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    w = (x2 - x1).clamp(min=0.0)
    h = (y2 - y1).clamp(min=0.0)
    return torch.stack([cx, cy, w, h], dim=-1).clamp(0.0, 1.0)


def cxcywh_norm_to_xyxy_abs_torch(boxes, img_size):
    if boxes.numel() == 0:
        return boxes.new_zeros((0, 4))
    boxes = boxes.clamp(0.0, 1.0)
    cx, cy, w, h = boxes.unbind(-1)
    x1 = (cx - w / 2.0) * float(img_size)
    y1 = (cy - h / 2.0) * float(img_size)
    x2 = (cx + w / 2.0) * float(img_size)
    y2 = (cy + h / 2.0) * float(img_size)
    return torch.stack([x1, y1, x2, y2], dim=-1).clamp(0.0, float(img_size))


def box_iou_xyxy_torch(boxes1, boxes2, eps=1e-7):
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return boxes1.new_zeros((boxes1.shape[0], boxes2.shape[0]))
    lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    area1 = ((boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0))
    area2 = ((boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0))
    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=eps)


def greedy_unique_matching(cost, max_pairs=None):
    """Greedy bipartite matching over a cost matrix. Selection is non-differentiable by design."""
    if cost.numel() == 0:
        return []
    n, q = cost.shape
    limit = min(n, q) if max_pairs is None else min(n, q, int(max_pairs))
    order = torch.argsort(cost.detach().flatten())
    used_i, used_q, pairs = set(), set(), []
    for lin in order.tolist():
        i, j = divmod(int(lin), q)
        if i in used_i or j in used_q:
            continue
        used_i.add(i)
        used_q.add(j)
        pairs.append((i, j))
        if len(pairs) >= limit:
            break
    return pairs


@torch.no_grad()
def mine_pseudo_labels(teacher_outputs, cfg, processor=None, score_threshold=0.4):
    """Mine pseudo labels and preserve teacher query indices for CQC.

    Notebook 04 used processor.post_process_object_detection. Notebook 05 computes the
    same type of pseudo targets directly from logits/pred_boxes so that each pseudo box
    keeps its originating teacher query index. This index is needed by the CQC proxy.
    """
    logits = teacher_outputs.logits
    pred_boxes = teacher_outputs.pred_boxes.clamp(0.0, 1.0)
    device = logits.device
    batch = []

    probs = torch.sigmoid(logits)
    scores_all, labels_all = probs.max(dim=-1)

    for b in range(logits.shape[0]):
        scores = scores_all[b]
        labels = labels_all[b].long()
        boxes_cxcywh = pred_boxes[b]
        query_indices = torch.arange(boxes_cxcywh.shape[0], device=device)

        keep = (scores >= float(score_threshold)) & (labels >= 0) & (labels < int(cfg["num_classes"]))
        boxes_cxcywh = boxes_cxcywh[keep]
        scores = scores[keep]
        labels = labels[keep]
        query_indices = query_indices[keep]

        if scores.numel() > 0:
            boxes_xyxy = cxcywh_norm_to_xyxy_abs_torch(boxes_cxcywh, cfg["img_size"])
            wh = boxes_xyxy[:, 2:4] - boxes_xyxy[:, 0:2]
            keep = (wh[:, 0] >= cfg["min_box_size_px"]) & (wh[:, 1] >= cfg["min_box_size_px"])
            boxes_cxcywh, scores, labels, query_indices = boxes_cxcywh[keep], scores[keep], labels[keep], query_indices[keep]

        if scores.numel() > 0 and nms is not None:
            boxes_xyxy = cxcywh_norm_to_xyxy_abs_torch(boxes_cxcywh, cfg["img_size"])
            keep_all = []
            for cls in labels.unique():
                idx = torch.where(labels == cls)[0]
                keep_idx = nms(boxes_xyxy[idx], scores[idx], cfg["nms_iou_threshold"])
                keep_all.append(idx[keep_idx])
            if keep_all:
                keep_all = torch.cat(keep_all)
                keep_all = keep_all[torch.argsort(scores[keep_all], descending=True)]
                boxes_cxcywh, scores, labels, query_indices = (
                    boxes_cxcywh[keep_all], scores[keep_all], labels[keep_all], query_indices[keep_all]
                )

        max_pseudo = int(cfg.get("max_pseudo_per_image", 0))
        if max_pseudo > 0 and scores.numel() > max_pseudo:
            idx = torch.topk(scores, k=max_pseudo).indices
            boxes_cxcywh, scores, labels, query_indices = boxes_cxcywh[idx], scores[idx], labels[idx], query_indices[idx]

        batch.append({
            "boxes": boxes_cxcywh.detach(),
            "class_labels": labels.detach(),
            "scores": scores.detach(),
            "query_indices": query_indices.detach(),
        })
    return batch


def pseudo_to_hf_labels(pseudo):
    return [
        {"boxes": p["boxes"], "class_labels": p["class_labels"]}
        for p in pseudo
    ]


@torch.no_grad()
def cpm_cost_based_pseudo_label_mining(pseudo_batch, student_outputs, cfg):
    """Cost-based Pseudo Label Mining proxy for CQC.

    The Semi-DETR paper mines reliable pseudo boxes for consistency training by
    clustering matching costs. This HF-compatible version:
    - computes a matching cost between each teacher pseudo box and student queries;
    - uses a 2-component Gaussian Mixture Model over the batch-level cost distribution;
    - keeps lower-cost pseudo boxes as CQC anchors.

    Returned pseudo boxes keep the original fields: boxes, class_labels, scores, query_indices.

    Implementation note: CPM cost is forced to float32 because AMP/autocast may produce
    half-precision student outputs, while teacher pseudo boxes are float32. torch.cdist
    requires both inputs to have the same dtype.
    """
    if not cfg.get("cpm_enabled", True):
        initial = sum(int(p.get("boxes", torch.empty(0)).shape[0]) for p in pseudo_batch)
        return pseudo_batch, {
            "initial": initial,
            "kept": initial,
            "keep_rate": 1.0 if initial > 0 else 0.0,
            "threshold": float("nan"),
            "fallback": 0,
            "method": "disabled",
        }

    student_logits = student_outputs.logits.detach().float()
    student_boxes = student_outputs.pred_boxes.detach().float().clamp(0.0, 1.0)
    device = student_logits.device

    records = []
    total_initial = 0

    for b, pseudo in enumerate(pseudo_batch):
        boxes = pseudo.get("boxes")
        labels = pseudo.get("class_labels")
        if boxes is None or labels is None or boxes.numel() == 0:
            continue

        boxes = boxes.detach().to(device=device, dtype=torch.float32).clamp(0.0, 1.0)
        labels = labels.detach().to(device=device, dtype=torch.long)
        total_initial += int(boxes.shape[0])

        s_probs = torch.sigmoid(student_logits[b]).float()
        s_boxes = student_boxes[b].float()

        cls_cost = (1.0 - s_probs[:, labels].transpose(0, 1).clamp(0.0, 1.0)).float()
        l1_cost = torch.cdist(boxes.float(), s_boxes.float(), p=1)
        iou = box_iou_xyxy_torch(
            cxcywh_norm_to_xyxy_abs_torch(boxes, cfg["img_size"]),
            cxcywh_norm_to_xyxy_abs_torch(s_boxes.float(), cfg["img_size"]),
        )
        cost = (
            float(cfg.get("cpm_cls_cost", 1.0)) * cls_cost.float()
            + float(cfg.get("cpm_l1_cost", 2.0)) * l1_cost
            + float(cfg.get("cpm_iou_cost", 2.0)) * (1.0 - iou)
        )

        pairs = greedy_unique_matching(cost, max_pairs=boxes.shape[0])
        for anchor_idx, query_idx in pairs:
            records.append({
                "batch_index": b,
                "anchor_index": int(anchor_idx),
                "query_index": int(query_idx),
                "cost": float(cost[anchor_idx, query_idx].detach().cpu()),
            })

    if total_initial == 0 or len(records) == 0:
        empty = []
        for pseudo in pseudo_batch:
            item = {}
            for k, v in pseudo.items():
                if torch.is_tensor(v) and v.ndim >= 1:
                    item[k] = v[:0]
                else:
                    item[k] = v
            empty.append(item)
        return empty, {
            "initial": int(total_initial),
            "kept": 0,
            "keep_rate": 0.0,
            "threshold": float("nan"),
            "fallback": 1,
            "method": "empty",
        }

    costs = np.asarray([r["cost"] for r in records], dtype=np.float32)
    strategy = str(cfg.get("cpm_strategy", "gmm_low_cluster"))
    fallback = 0
    threshold = float("nan")

    if (
        GaussianMixture is not None
        and len(costs) >= int(cfg.get("cpm_min_gmm_samples", 4))
        and float(np.std(costs)) > 1e-8
        and strategy in {"gmm_low_cluster", "reliable_center", "midpoint"}
    ):
        try:
            gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=int(cfg.get("seed", 42)))
            cluster_ids = gmm.fit_predict(costs.reshape(-1, 1))
            means = gmm.means_.reshape(-1)
            low_cluster = int(np.argmin(means))
            high_cluster = int(np.argmax(means))

            if strategy == "gmm_low_cluster":
                keep_mask = cluster_ids == low_cluster
                threshold = float(np.max(costs[keep_mask])) if np.any(keep_mask) else float(np.min(costs))
            elif strategy == "reliable_center":
                threshold = float(means[low_cluster])
                keep_mask = costs <= threshold
            else:  # midpoint
                threshold = float((means[low_cluster] + means[high_cluster]) / 2.0)
                keep_mask = costs <= threshold
        except Exception:
            fallback = 1
            threshold = float(np.quantile(costs, float(cfg.get("cpm_fallback_quantile", 0.60))))
            keep_mask = costs <= threshold
    else:
        fallback = 1
        threshold = float(np.quantile(costs, float(cfg.get("cpm_fallback_quantile", 0.60))))
        keep_mask = costs <= threshold

    if not np.any(keep_mask):
        keep_mask[int(np.argmin(costs))] = True

    keep_by_image = defaultdict(list)
    for rec, keep in zip(records, keep_mask):
        if bool(keep):
            keep_by_image[rec["batch_index"]].append(rec["anchor_index"])

    if bool(cfg.get("cpm_keep_at_least_one_per_image", True)):
        best_by_image = {}
        for rec in records:
            b = rec["batch_index"]
            if b not in best_by_image or rec["cost"] < best_by_image[b]["cost"]:
                best_by_image[b] = rec
        for b, rec in best_by_image.items():
            if len(keep_by_image[b]) == 0:
                keep_by_image[b].append(rec["anchor_index"])

    filtered = []
    kept_total = 0
    for b, pseudo in enumerate(pseudo_batch):
        boxes = pseudo.get("boxes")
        if boxes is None or boxes.numel() == 0:
            item = {}
            for k, v in pseudo.items():
                if torch.is_tensor(v) and v.ndim >= 1:
                    item[k] = v[:0]
                else:
                    item[k] = v
            filtered.append(item)
            continue

        idx_list = sorted(set(int(i) for i in keep_by_image.get(b, [])))
        if len(idx_list) == 0:
            keep_idx = torch.empty(0, dtype=torch.long, device=boxes.device)
        else:
            keep_idx = torch.tensor(idx_list, dtype=torch.long, device=boxes.device)
        kept_total += int(keep_idx.numel())

        n = int(boxes.shape[0])
        item = {}
        for k, v in pseudo.items():
            if torch.is_tensor(v) and v.ndim >= 1 and v.shape[0] == n:
                item[k] = v[keep_idx]
            else:
                item[k] = v
        filtered.append(item)

    return filtered, {
        "initial": int(total_initial),
        "kept": int(kept_total),
        "keep_rate": float(kept_total / max(1, total_initial)),
        "threshold": float(threshold),
        "fallback": int(fallback),
        "method": strategy,
    }


def cqc_proxy_loss(teacher_outputs, student_outputs, pseudo_batch, cfg):
    """Cross-view Query Consistency proxy.

    Paper-level CQC injects RoI-derived cross-view query embeddings into the decoder.
    This HF-compatible proxy uses pseudo boxes as semantic anchors:
    1) teacher weak-view queries produce pseudo boxes;
    2) student strong-view queries are matched to the same pseudo boxes;
    3) matched teacher/student query predictions are forced to be similar.
    """
    student_logits = student_outputs.logits
    student_boxes = student_outputs.pred_boxes.clamp(0.0, 1.0)
    teacher_logits = teacher_outputs.logits.detach()

    total = student_logits.new_tensor(0.0)
    valid_images = 0
    total_matches = 0

    for b, pseudo in enumerate(pseudo_batch):
        boxes_t = pseudo.get("boxes")
        labels = pseudo.get("class_labels")
        q_teacher = pseudo.get("query_indices")
        if boxes_t is None or labels is None or q_teacher is None or boxes_t.numel() == 0:
            continue

        # Limit CQC anchors for memory/time stability.
        max_cqc = int(cfg.get("cqc_max_boxes_per_image", 30))
        if boxes_t.shape[0] > max_cqc:
            scores = pseudo.get("scores")
            if scores is not None and scores.numel() == boxes_t.shape[0]:
                keep = torch.topk(scores, k=max_cqc).indices
            else:
                keep = torch.arange(max_cqc, device=boxes_t.device)
            boxes_t = boxes_t[keep]
            labels = labels[keep]
            q_teacher = q_teacher[keep]

        boxes_t = boxes_t.detach().to(student_logits.device)
        labels = labels.detach().long().to(student_logits.device)
        q_teacher = q_teacher.detach().long().to(student_logits.device)

        s_probs = torch.sigmoid(student_logits[b]).float()
        s_boxes = student_boxes[b].float()
        t_probs = torch.sigmoid(teacher_logits[b, q_teacher]).detach()

        # Match each pseudo box to a strong-view student query by class score + box proximity + IoU.
        cls_cost = 1.0 - s_probs[:, labels].transpose(0, 1).clamp(0.0, 1.0)
        l1_cost = torch.cdist(boxes_t, s_boxes, p=1)
        iou = box_iou_xyxy_torch(
            cxcywh_norm_to_xyxy_abs_torch(boxes_t, cfg["img_size"]),
            cxcywh_norm_to_xyxy_abs_torch(s_boxes.float(), cfg["img_size"]),
        )
        cost = (
            float(cfg.get("cqc_match_cls_cost", 1.0)) * cls_cost
            + float(cfg.get("cqc_match_l1_cost", 2.0)) * l1_cost
            + float(cfg.get("cqc_match_iou_cost", 2.0)) * (1.0 - iou)
        )
        pairs = greedy_unique_matching(cost, max_pairs=boxes_t.shape[0])
        if len(pairs) == 0:
            continue

        anchor_idx = torch.tensor([p[0] for p in pairs], device=student_logits.device, dtype=torch.long)
        student_q_idx = torch.tensor([p[1] for p in pairs], device=student_logits.device, dtype=torch.long)

        s_vec = s_probs[student_q_idx]
        t_vec = t_probs[anchor_idx]
        cls_cons = (1.0 - F.cosine_similarity(s_vec, t_vec, dim=-1)).mean()
        box_cons = F.l1_loss(s_boxes[student_q_idx], boxes_t[anchor_idx], reduction="mean")

        total = total + float(cfg.get("cqc_cls_weight", 1.0)) * cls_cons + float(cfg.get("cqc_box_weight", 2.0)) * box_cons
        valid_images += 1
        total_matches += len(pairs)

    if valid_images == 0:
        return student_logits.new_tensor(0.0), 0
    return total / valid_images, int(total_matches)


class EMATeacherStudent(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.student = build_detector(cfg)
        self.teacher = copy.deepcopy(self.student)
        for p in self.teacher.parameters():
            p.requires_grad_(False)
        self.teacher.eval()
        self.ema_momentum = float(cfg["ema_momentum"])

    @torch.no_grad()
    def copy_student_to_teacher(self):
        self.teacher.load_state_dict(self.student.state_dict())
        self.teacher.eval()

    @torch.no_grad()
    def ema_update(self):
        m = self.ema_momentum
        for tp, sp in zip(self.teacher.parameters(), self.student.parameters()):
            tp.data.mul_(m).add_(sp.data, alpha=1.0 - m)
        for tb, sb in zip(self.teacher.buffers(), self.student.buffers()):
            tb.copy_(sb)
        self.teacher.eval()

    def forward_student(self, pixel_values, pixel_mask=None, labels=None):
        return self.student(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)

    @torch.no_grad()
    def forward_teacher(self, pixel_values, pixel_mask=None):
        self.teacher.eval()
        return self.teacher(pixel_values=pixel_values, pixel_mask=pixel_mask)


In [ ]:
model = EMATeacherStudent(CFG).to(DEVICE)
optimizer = build_optimizer(model.student, CFG)
scheduler = build_scheduler(optimizer, CFG)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda")
unlabeled_iter = infinite_loader(unlabeled_loader)

with contextlib.redirect_stdout(io.StringIO()):
    VALID_COCO_GT = build_coco_gt_from_dataset(valid_ds, name="valid_cache")

print(f"Student trainable params: {sum(p.numel() for p in model.student.parameters() if p.requires_grad) / 1e6:.1f}M")
print("Checkpoint criterion: best validation AP (COCO AP@[.50:.95]), not AP50")
print("Experiment: Full Semi-DETR-style SHM + CQC + CPM. SHM/CQC/CPM are HF-compatible approximations/proxies.")

history = []
best_ap = -1.0
best_epoch = -1
best_ckpt = OUT_DIR / "best_model.pt"
last_ckpt = OUT_DIR / "last_model.pt"

header = (
    f"{'Ep':>4} {'stage':>13} {'loss':>8} {'sup':>7} {'unsup':>7} {'cqc':>7} "
    f"{'pseudo':>8} {'cM':>5} {'cpmK':>6} {'cpm%':>6} {'tgtx':>5} {'thr':>6} {'w_u':>5} {'w_c':>5} "
    f"{'lr':>10} {'grad':>7} {'time':>7} {'val_AP':>8} {'AP50':>7} {'AP75':>7} {'APs':>7}"
)
print("-" * len(header))
print(header)
print("-" * len(header))

for epoch in range(1, CFG["epochs"] + 1):
    model.student.train()
    t0 = time.time()
    stage = get_shm_stage(epoch, CFG)
    use_unsup = epoch > CFG["burn_in_epochs"] and len(unlabeled_train_ds) > 0
    pseudo_thr = get_pseudo_threshold(epoch, CFG)
    unsup_w = get_unsup_weight(epoch, CFG)
    cqc_w = get_cqc_weight(epoch, CFG)

    running = defaultdict(float)
    n_unlabeled_images = 0
    grad_norm_sum = 0.0
    grad_steps = 0
    optimizer.zero_grad(set_to_none=True)

    for step, labeled_batch in enumerate(labeled_loader):
        pv_lab = labeled_batch["pixel_values"].to(DEVICE)
        pm_lab = labeled_batch.get("pixel_mask")
        pm_lab = pm_lab.to(DEVICE) if pm_lab is not None else None

        labels_lab_o2o = move_labels_to_device(labeled_batch["labels"], DEVICE)
        labels_lab = maybe_apply_shm_o2m(
            labels_lab_o2o,
            epoch,
            CFG,
            apply=bool(CFG.get("shm_apply_to_supervised", True)),
        )

        with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda"):
            sup_out = model.forward_student(pv_lab, pm_lab, labels_lab)
            sup_loss = sup_out.loss
            unsup_loss = sup_loss.new_tensor(0.0)
            cqc_loss_val = sup_loss.new_tensor(0.0)

        pseudo_count = 0
        cqc_match_count = 0
        cpm_kept_count = 0
        cpm_initial_count = 0
        cpm_keep_rate = 0.0
        cpm_threshold = float("nan")
        cpm_fallback = 0
        o2m_target_multiplier = 1.0
        unsup_losses = []
        cqc_losses = []
        o2m_multipliers = []
        cpm_thresholds = []

        if use_unsup:
            for _ in range(int(CFG.get("unlabeled_steps_per_labeled", 1))):
                unlabeled_batch = next(unlabeled_iter)
                pv_w = unlabeled_batch["weak_pixel_values"].to(DEVICE)
                pm_w = unlabeled_batch.get("weak_pixel_mask")
                pm_w = pm_w.to(DEVICE) if pm_w is not None else None
                pv_s = unlabeled_batch["strong_pixel_values"].to(DEVICE)
                pm_s = unlabeled_batch.get("strong_pixel_mask")
                pm_s = pm_s.to(DEVICE) if pm_s is not None else None
                n_unlabeled_images += pv_w.shape[0]

                teacher_out = model.forward_teacher(pv_w, pm_w)
                pseudo = mine_pseudo_labels(teacher_out, CFG, processor, score_threshold=pseudo_thr)
                pseudo_count_i = sum(len(x["boxes"]) for x in pseudo)
                pseudo_count += pseudo_count_i

                if pseudo_count_i > 0:
                    pseudo_labels_o2o = move_labels_to_device(pseudo_to_hf_labels(pseudo), DEVICE)
                    pseudo_targets_before = count_targets(pseudo_labels_o2o)
                    pseudo_labels = maybe_apply_shm_o2m(
                        pseudo_labels_o2o,
                        epoch,
                        CFG,
                        apply=bool(CFG.get("shm_apply_to_unsupervised", True)),
                    )
                    pseudo_targets_after = count_targets(pseudo_labels)
                    o2m_multipliers.append(pseudo_targets_after / max(1, pseudo_targets_before))

                    with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda"):
                        unsup_out = model.forward_student(pv_s, pm_s, pseudo_labels)
                        unsup_losses.append(unsup_out.loss)

                    pseudo_for_cqc, cpm_stats = cpm_cost_based_pseudo_label_mining(pseudo, unsup_out, CFG)
                    cpm_initial_count += int(cpm_stats.get("initial", 0))
                    cpm_kept_count += int(cpm_stats.get("kept", 0))
                    cpm_fallback += int(cpm_stats.get("fallback", 0))
                    thr_val = float(cpm_stats.get("threshold", float("nan")))
                    if math.isfinite(thr_val):
                        cpm_thresholds.append(thr_val)

                    with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda"):
                        cqc_loss_i, cqc_match_i = cqc_proxy_loss(teacher_out, unsup_out, pseudo_for_cqc, CFG)
                        cqc_losses.append(cqc_loss_i)
                        cqc_match_count += int(cqc_match_i)

            if unsup_losses:
                unsup_loss = torch.stack(unsup_losses).mean()
            if cqc_losses:
                cqc_loss_val = torch.stack(cqc_losses).mean()
            if o2m_multipliers:
                o2m_target_multiplier = float(sum(o2m_multipliers) / len(o2m_multipliers))
            if cpm_initial_count > 0:
                cpm_keep_rate = cpm_kept_count / max(1, cpm_initial_count)
            if cpm_thresholds:
                cpm_threshold = float(sum(cpm_thresholds) / len(cpm_thresholds))

        with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda"):
            full_loss = sup_loss + unsup_w * unsup_loss + cqc_w * cqc_loss_val
            total_loss = full_loss / CFG["grad_accum_steps"]

        scaler.scale(total_loss).backward()

        do_step = ((step + 1) % CFG["grad_accum_steps"] == 0) or ((step + 1) == len(labeled_loader))
        if do_step:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.student.parameters(), CFG["clip_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            grad_norm_sum += float(grad_norm)
            grad_steps += 1

            if epoch <= CFG["burn_in_epochs"]:
                model.copy_student_to_teacher()
            else:
                model.ema_update()

        running["loss"] += float(full_loss.detach().cpu())
        running["sup"] += float(sup_loss.detach().cpu())
        running["unsup"] += float(unsup_loss.detach().cpu())
        running["cqc"] += float(cqc_loss_val.detach().cpu())
        running["pseudo"] += float(pseudo_count)
        running["cqc_matches"] += float(cqc_match_count)
        running["cpm_initial"] += float(cpm_initial_count)
        running["cpm_kept"] += float(cpm_kept_count)
        running["cpm_keep_rate_sum"] += float(cpm_keep_rate)
        running["cpm_keep_rate_n"] += 1.0 if cpm_initial_count > 0 else 0.0
        if not math.isnan(cpm_threshold):
            running["cpm_threshold_sum"] += float(cpm_threshold)
            running["cpm_threshold_n"] += 1.0
        running["cpm_fallback"] += float(cpm_fallback)
        running["tgt_multiplier"] += float(o2m_target_multiplier)

    scheduler.step()
    n = max(1, len(labeled_loader))
    pseudo_per_img = running["pseudo"] / max(1, n_unlabeled_images) if use_unsup else 0.0
    cqc_matches_per_img = running["cqc_matches"] / max(1, n_unlabeled_images) if use_unsup else 0.0
    cpm_kept_per_img = running["cpm_kept"] / max(1, n_unlabeled_images) if use_unsup else 0.0
    cpm_keep_rate_epoch = running["cpm_kept"] / max(1.0, running["cpm_initial"]) if use_unsup else 0.0
    cpm_threshold_epoch = running["cpm_threshold_sum"] / max(1.0, running["cpm_threshold_n"]) if running["cpm_threshold_n"] > 0 else float("nan")
    train_stats = {
        "epoch": epoch,
        "stage": stage,
        "loss": running["loss"] / n,
        "sup": running["sup"] / n,
        "unsup": running["unsup"] / n,
        "cqc": running["cqc"] / n,
        "pseudo_per_img": float(pseudo_per_img),
        "cqc_matches_per_img": float(cqc_matches_per_img),
        "cpm_kept_per_img": float(cpm_kept_per_img),
        "cpm_keep_rate": float(cpm_keep_rate_epoch),
        "cpm_threshold": float(cpm_threshold_epoch),
        "cpm_fallback_steps": int(running["cpm_fallback"]),
        "o2m_target_multiplier": running["tgt_multiplier"] / n,
        "pseudo_threshold": float(pseudo_thr),
        "unsup_weight": float(unsup_w),
        "cqc_weight": float(cqc_w),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "grad_norm": grad_norm_sum / max(1, grad_steps),
        "elapsed_min": (time.time() - t0) / 60.0,
    }

    metrics = None
    saved_flag = ""
    if epoch % CFG["eval_every_n_epochs"] == 0 or epoch == CFG["epochs"]:
        metrics = evaluate_coco(model.student, valid_loader, valid_ds, DEVICE, coco_gt=VALID_COCO_GT, desc=f"val_ep{epoch}")
        ckpt = {
            "epoch": epoch,
            "student_state": model.student.state_dict(),
            "teacher_state": model.teacher.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_ap": max(best_ap, metrics["AP"]),
            "config": CFG,
            "history": history,
        }
        torch.save(ckpt, last_ckpt)
        if metrics["AP"] > best_ap:
            best_ap = metrics["AP"]
            best_epoch = epoch
            torch.save(ckpt, best_ckpt)
            saved_flag = "*"

    row = {**train_stats}
    if metrics is not None:
        row.update(metrics)
    history.append(row)

    ap = f"{metrics['AP']:.3f}" if metrics else "—"
    ap50 = f"{metrics['AP50']:.3f}" if metrics else "—"
    ap75 = f"{metrics['AP75']:.3f}" if metrics else "—"
    aps = f"{metrics['APs']:.3f}" if metrics else "—"
    print(
        f"{epoch:4d} {stage:>13} {train_stats['loss']:8.4f} {train_stats['sup']:7.4f} "
        f"{train_stats['unsup']:7.4f} {train_stats['cqc']:7.4f} {train_stats['pseudo_per_img']:8.2f} "
        f"{train_stats['cqc_matches_per_img']:5.2f} {train_stats['cpm_kept_per_img']:6.2f} {train_stats['cpm_keep_rate']:6.2f} "
        f"{train_stats['o2m_target_multiplier']:5.2f} "
        f"{train_stats['pseudo_threshold']:6.3f} {train_stats['unsup_weight']:5.2f} {train_stats['cqc_weight']:5.2f} "
        f"{train_stats['lr']:10.2e} {train_stats['grad_norm']:7.2f} {train_stats['elapsed_min']:6.1f}m "
        f"{ap:>8} {ap50:>7} {ap75:>7} {aps:>7} {saved_flag}"
    )

with open(OUT_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)

print("-" * len(header))
print(f"Training complete. Best validation AP={best_ap:.4f} at epoch {best_epoch}")


In [ ]:
ckpt = safe_torch_load(best_ckpt if best_ckpt.exists() else last_ckpt, DEVICE)
model.student.load_state_dict(ckpt["student_state"])
model.teacher.load_state_dict(ckpt["teacher_state"])
print(f"Loaded checkpoint: epoch={ckpt['epoch']} | best_AP={ckpt.get('best_ap', best_ap):.4f}")

with contextlib.redirect_stdout(io.StringIO()):
    TEST_COCO_GT = build_coco_gt_from_dataset(test_ds, name="test_cache")

val_metrics = evaluate_coco(model.student, valid_loader, valid_ds, DEVICE, coco_gt=VALID_COCO_GT, desc="final_val")
test_metrics = evaluate_coco(model.student, test_loader, test_ds, DEVICE, coco_gt=TEST_COCO_GT, desc="final_test")

final_metrics = {
    "method": "full_semi_detr_shm_cqc_cpm_10pct_100e",
    "best_epoch": int(ckpt["epoch"]),
    "best_val_ap": float(ckpt.get("best_ap", best_ap)),
    "val": val_metrics,
    "test": test_metrics,
    "config": CFG,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print(json.dumps({
    "best_epoch": final_metrics["best_epoch"],
    "val": {k: round(v, 4) for k, v in val_metrics.items() if k != "per_class_AP50"},
    "test": {k: round(v, 4) for k, v in test_metrics.items() if k != "per_class_AP50"},
}, indent=2))

In [ ]:
# Plots tối giản để kiểm tra hội tụ, pseudo-label, SHM stage, CPM và metric.
if len(history):
    epochs = [h["epoch"] for h in history]
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, [h["loss"] for h in history], label="total")
    plt.plot(epochs, [h["sup"] for h in history], label="supervised")
    plt.plot(epochs, [h["unsup"] for h in history], label="unsupervised")
    plt.plot(epochs, [h.get("cqc", 0.0) for h in history], label="CQC")
    plt.axvline(CFG["burn_in_epochs"], linestyle="--", label="burn-in end")
    plt.axvline(CFG["shm_stage1_epochs"], linestyle=":", label="SHM O2M end")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title("Full Semi-DETR-style SHM + CQC + CPM training losses")
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "plot_losses.png", dpi=140); plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, [h["pseudo_per_img"] for h in history], label="pseudo/img")
    plt.plot(epochs, [h["pseudo_threshold"] for h in history], label="pseudo threshold")
    plt.plot(epochs, [h["unsup_weight"] for h in history], label="unsup weight")
    plt.plot(epochs, [h.get("cqc_weight", 0.0) for h in history], label="CQC weight")
    plt.plot(epochs, [h.get("cpm_keep_rate", 0.0) for h in history], label="CPM keep rate")
    plt.axvline(CFG["burn_in_epochs"], linestyle="--", label="burn-in end")
    plt.axvline(CFG["shm_stage1_epochs"], linestyle=":", label="SHM O2M end")
    plt.xlabel("Epoch"); plt.title("Full Semi-DETR-style pseudo-label / CPM schedule")
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "plot_pseudo_schedule.png", dpi=140); plt.show()

    eval_rows = [h for h in history if "AP" in h]
    if eval_rows:
        ev = [h["epoch"] for h in eval_rows]
        plt.figure(figsize=(8, 4))
        plt.plot(ev, [h["AP"] for h in eval_rows], "o-", label="AP")
        plt.plot(ev, [h["AP50"] for h in eval_rows], "s--", label="AP50")
        plt.plot(ev, [h["AP75"] for h in eval_rows], "^--", label="AP75")
        plt.axvline(CFG["burn_in_epochs"], linestyle="--", label="burn-in end")
        plt.axvline(CFG["shm_stage1_epochs"], linestyle=":", label="SHM O2M end")
        plt.xlabel("Epoch"); plt.ylabel("COCO AP")
        plt.title("Full Semi-DETR-style SHM + CQC + CPM validation metrics")
        plt.legend(); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(OUT_DIR / "plot_val_metrics.png", dpi=140); plt.show()